# 🎾 [TennisLab] 올림픽공원 19개 테니스 코트 환경 동역학 분석 노트
> **한국체육산업개발 공공데이터와 기상청 관측 통계를 결합한 스포츠 데이터 사이언스 연구**

본 분석 노트는 서울 올림픽공원 테니스장 19개 코트의 방위각 메타데이터와 연간 8,760건의 기상 관측 데이터를 결합하여,
**풍향, 풍속, 기온** 등의 환경 요인이 테니스공의 **유효 비거리, 착지점 편차, 볼 바운스 높이**에 미치는 영향을 통계적/물리적으로 규명합니다.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드
df_weather = pd.read_csv('../data/olympic_weather_history.csv')
with open('../data/olympic_courts.json', 'r', encoding='utf-8') as f:
    courts = json.load(f)

print(f"기상 데이터 수집 건수: {len(df_weather):,}건")
print(f"올림픽공원 코트 수: {len(courts)}개 면")

## 1. 계절별 19개 코트 바람 취약도 히트맵 분석
실외 14개 코트와 센터코트, 실내 4개 코트의 계절별 체감 풍속을 분석합니다.

In [2]:
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
season_kr = {'Spring': '봄', 'Summer': '여름', 'Autumn': '가을', 'Winter': '겨울'}

heatmap_data = []
for c in courts:
    row = []
    for s in seasons:
        if c['has_roof']:
            row.append(0.0)
        else:
            df_s = df_weather[df_weather['season'] == s]
            rel_angles = (df_s['wind_dir_deg'] - c['orientation_deg']) % 360
            eff_winds = df_s['wind_speed_ms'] * np.abs(np.cos(np.radians(rel_angles)))
            row.append(round(eff_winds.mean(), 2))
    heatmap_data.append(row)

df_heat = pd.DataFrame(heatmap_data, index=[f"코트 {c['court_id']}" for c in courts], columns=[season_kr[s] for s in seasons])
df_heat.head(10)

## 2. 풍속에 따른 스트로크 비거리 편차 및 아웃(Out) 확률 분포
90km/h 스트로크 기준으로 풍속(0~6m/s)에 따른 비거리 증가 및 실책 위험도를 시뮬레이션합니다.

In [3]:
# 생성된 고화질 차트 불러오기
from IPython.display import Image
Image(filename='../docs/images/trajectory_deviation_dist.png')

## 3. 기온에 따른 하드코트 볼 바운스 반발계수(COR) 및 스피드 변화

In [4]:
Image(filename='../docs/images/court_bounce_pace.png')